# Necessary Subcircuit Semantic Analysis

Post-hoc semantic analysis for `select_necessary_collapsed_edges.py` outputs. This notebook reads the selected necessary-edge reports, labels structural edge roles, builds component semantic summaries, and provides opt-in activation/readout/PCA analysis for the selected necessary subcircuits. It does not run EAP attribution or rediscover edges.


In [15]:
from __future__ import annotations

import gc
import json
import os
import re
import sys
from datetime import date
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
try:
    from IPython.display import HTML, display
except Exception:
    class HTML(str):
        pass

    display = print

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "scripts":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "animacy-circuit").is_dir():
    PROJECT_ROOT = PROJECT_ROOT / "animacy-circuit"

SCRIPT_ROOT = PROJECT_ROOT / "scripts" / "executable"
if str(SCRIPT_ROOT) not in sys.path:
    sys.path.insert(0, str(SCRIPT_ROOT))

try:
    from circuit_finder_core import canonical_model_name, eap_node_metadata, safe_model_name
except Exception as exc:
    CORE_IMPORT_ERROR = exc

    def canonical_model_name(model_name: str) -> str:
        return model_name

    def safe_model_name(model_name: str) -> str:
        return str(model_name).replace("/", "_")

    def eap_node_metadata(node_name: str) -> dict[str, Any]:
        if node_name == "input":
            return {"kind": "input", "layer": -1, "head": None}
        if node_name == "logits":
            return {"kind": "logits", "layer": -1, "head": None}
        mlp_match = re.fullmatch(r"m(\d+)", str(node_name))
        if mlp_match:
            return {"kind": "mlp", "layer": int(mlp_match.group(1)), "head": None}
        attn_match = re.fullmatch(r"a(\d+)\.h(\d+)", str(node_name))
        if attn_match:
            return {"kind": "attn", "layer": int(attn_match.group(1)), "head": int(attn_match.group(2))}
        return {"kind": "other", "layer": 0, "head": None}

RESULTS_ROOT = PROJECT_ROOT / "results" / "eap_ig_localization"
preferred_reports = sorted(RESULTS_ROOT.glob("necessary_edge_expansion_main_original_20_50_*"))
fallback_reports = sorted(RESULTS_ROOT.glob("necessary_edge_expansion_*"))
REPORT_DIR = (preferred_reports or fallback_reports)[-1]

N_EXAMPLES = 500
DEBUG_N_EXAMPLES = 32
BATCH_SIZE = 4
MODEL_SLUGS = None  # None means all selected slots in the report.
SEEDS = [42]  # Default to the canonical seed; set to None to restore all selected seeds.
LOAD_ACTIVATION_RESULTS = True
ACTIVATION_RESULTS_DIR = "animacy-circuit/results/eap_ig_localization/necessary_semantics_activations_2026-06-21" # None means latest results/eap_ig_localization/necessary_semantics_activations_*.
EXPORT_CSV = False
EXPORT_DIR = RESULTS_ROOT / f"necessary_semantics_{date.today().isoformat()}"


def resolve_existing_path(value: str | Path | None) -> Path | None:
    if value is None:
        return None
    path = Path(value).expanduser()
    if path.is_absolute():
        return path if path.exists() else path
    candidates = [
        Path.cwd().resolve() / path,
        PROJECT_ROOT / path,
        PROJECT_ROOT.parent / path,
        PROJECT_ROOT / "results" / "eap_ig_localization" / path.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[-1]


pio.renderers.default = "notebook_connected"


def show_fig(fig, *, height: int | None = None) -> None:
    if height is not None:
        fig.update_layout(height=height)
    try:
        fig.show(renderer="notebook_connected")
    except Exception:
        display(HTML(fig.to_html(full_html=False, include_plotlyjs=True)))

REPORT_DIR


PosixPath('/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/eap_ig_localization/necessary_edge_expansion_main_original_20_50_2026-06-20')

## Direct Activation Plots

This cell directly loads the precomputed CSV bundle and immediately renders the plots. It does not depend on later notebook state.


In [16]:
from __future__ import annotations

# Direct, self-contained activation-result loading and plotting.
DIRECT_RESULTS_DIR = resolve_existing_path(ACTIVATION_RESULTS_DIR)
if DIRECT_RESULTS_DIR is None or not DIRECT_RESULTS_DIR.exists():
    raise FileNotFoundError(f"Activation results directory not found: {ACTIVATION_RESULTS_DIR!r} resolved to {DIRECT_RESULTS_DIR}")

attention_semantics = pd.read_csv(DIRECT_RESULTS_DIR / "attention_semantics.csv")
readout_semantics = pd.read_csv(DIRECT_RESULTS_DIR / "readout_semantics.csv")
activation_geometry_summary = pd.read_csv(DIRECT_RESULTS_DIR / "activation_geometry_summary.csv")
activation_pca_projection = pd.read_csv(DIRECT_RESULTS_DIR / "activation_pca_projection.csv")

print(f"Loaded activation results from: {DIRECT_RESULTS_DIR}")
print(f"attention_semantics: {attention_semantics.shape}")
print(f"readout_semantics: {readout_semantics.shape}")
print(f"activation_geometry_summary: {activation_geometry_summary.shape}")
print(f"activation_pca_projection: {activation_pca_projection.shape}")

if attention_semantics.empty or readout_semantics.empty or activation_geometry_summary.empty or activation_pca_projection.empty:
    raise ValueError("One or more activation tables are empty; stopping instead of silently skipping plots.")

# Plot 1: attention mass to key positions.
attention_plot = attention_semantics.copy()
attention_plot["key_position_label"] = attention_plot["key_position_label"].replace({"final": "the"})
attention_order = ["BOS", "The", "patient", "was", "verb", "by", "the"]
attention_plot = (
    attention_plot.groupby(["model_slug", "condition", "key_position_label"], dropna=False)
    .agg(
        attention_mass_mean=("attention_mass_mean", "mean"),
        attention_mass_std=("attention_mass_std", "mean"),
        example_count=("example_count", "mean"),
    )
    .reset_index()
)
attention_index = pd.MultiIndex.from_product(
    [
        sorted(attention_plot["model_slug"].unique()),
        sorted(attention_plot["condition"].unique()),
        attention_order,
    ],
    names=["model_slug", "condition", "key_position_label"],
)
attention_plot = (
    attention_plot.set_index(["model_slug", "condition", "key_position_label"])
    .reindex(attention_index)
    .reset_index()
)
attention_plot["observed"] = attention_plot["attention_mass_mean"].notna()
attention_fig = px.bar(
    attention_plot,
    x="key_position_label",
    y="attention_mass_mean",
    color="condition",
    barmode="group",
    facet_col="model_slug",
    facet_col_wrap=2,
    hover_data=["attention_mass_std", "example_count", "observed"],
    category_orders={"key_position_label": attention_order},
    title="Necessary attention heads: mean attention from final 'the' to sentence positions",
)
attention_fig.update_yaxes(matches=None)
attention_fig.update_layout(xaxis_title="Key position", yaxis_title="Mean attention mass", height=760)
show_fig(attention_fig)
print("Direct plot cell renders only the attention-position figure. Use the later sections for readout, geometry, and PCA plots.")


Loaded activation results from: /gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/eap_ig_localization/necessary_semantics_activations_2026-06-21
attention_semantics: (288, 14)
readout_semantics: (720, 18)
activation_geometry_summary: (360, 18)
activation_pca_projection: (360000, 12)


Direct plot cell renders only the attention-position figure. Use the later sections for readout, geometry, and PCA plots.


## Load Necessary-Edge Report

The report directory supplies `necessary_budget_summary.csv`, `necessary_collapsed_edges.csv`, and `necessary_underlying_edges.csv`. The notebook keeps each slot's actual `selected_budget` instead of assuming every necessary subcircuit has the same number of collapsed edges. By default it focuses on `seed 42`; set `SEEDS = None` in the config cell to bring back all selected seeds.


In [17]:
from __future__ import annotations
SUMMARY_PATH = REPORT_DIR / "necessary_budget_summary.csv"
COLLAPSED_PATH = REPORT_DIR / "necessary_collapsed_edges.csv"
UNDERLYING_PATH = REPORT_DIR / "necessary_underlying_edges.csv"

summary = pd.read_csv(SUMMARY_PATH)
collapsed = pd.read_csv(COLLAPSED_PATH)
underlying = pd.read_csv(UNDERLYING_PATH)

REQUIRED_SUMMARY_COLUMNS = {
    "model_slug", "run_name", "sample_size", "seed", "status", "edge_path", "selected_budget"
}
REQUIRED_COLLAPSED_COLUMNS = {
    "model_slug", "run_name", "sample_size", "seed", "selected_budget",
    "collapsed_edge", "parent", "child", "signed_sum", "abs_score", "rank", "underlying_edge_count"
}
REQUIRED_UNDERLYING_COLUMNS = {
    "model_slug", "run_name", "sample_size", "seed", "selected_budget",
    "collapsed_rank", "collapsed_edge", "underlying_edge"
}

def require_columns(frame: pd.DataFrame, required: set[str], name: str) -> None:
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

require_columns(summary, REQUIRED_SUMMARY_COLUMNS, "necessary_budget_summary.csv")
require_columns(collapsed, REQUIRED_COLLAPSED_COLUMNS, "necessary_collapsed_edges.csv")
require_columns(underlying, REQUIRED_UNDERLYING_COLUMNS, "necessary_underlying_edges.csv")

selected_summary = summary[summary["status"].isin(["selected", "selected_from_partial"])].copy()
if MODEL_SLUGS is not None:
    selected_summary = selected_summary[selected_summary["model_slug"].isin(MODEL_SLUGS)].copy()
if SEEDS is not None:
    selected_summary = selected_summary[selected_summary["seed"].isin(SEEDS)].copy()

for frame in (selected_summary, collapsed, underlying):
    if "selected_budget" in frame.columns:
        frame["selected_budget"] = frame["selected_budget"].astype("Int64")

if MODEL_SLUGS is not None:
    collapsed = collapsed[collapsed["model_slug"].isin(MODEL_SLUGS)].copy()
    underlying = underlying[underlying["model_slug"].isin(MODEL_SLUGS)].copy()
if SEEDS is not None:
    collapsed = collapsed[collapsed["seed"].isin(SEEDS)].copy()
    underlying = underlying[underlying["seed"].isin(SEEDS)].copy()

if selected_summary.empty:
    raise ValueError("No selected necessary-edge slots were found in the configured report.")

SLOT_COLUMNS = ["model_slug", "run_name", "sample_size", "seed"]

def slot_label(frame: pd.DataFrame) -> pd.Series:
    return (
        frame["model_slug"].astype(str)
        + " / " + frame["run_name"].astype(str)
        + " / s" + frame["sample_size"].astype(str)
        + " seed " + frame["seed"].astype(str)
        + " / k=" + frame["selected_budget"].astype(str)
    )

selected_summary = selected_summary.assign(slot=slot_label(selected_summary))
collapsed = collapsed.assign(slot=slot_label(collapsed))
underlying = underlying.assign(slot=slot_label(underlying))

assert not collapsed.empty, "Selected report has no collapsed edges."

display(selected_summary.head())
display(collapsed.head())
display(underlying.head())


,model_slug,run_name,sample_size,seed,threshold,status,edge_path,eval_path,partial_eval_path,eval_source,selected_budget,selected_faithfulness,selected_accuracy,selected_collapsed_edge_count,selected_underlying_edge_count,slot
2,Qwen_Qwen3-4B,qwen_seed_stability,500,42,0.1,selected,animacy-circuit/results/eap_ig_localization/Qw...,animacy-circuit/results/eap_ig_localization/Qw...,animacy-circuit/results/eap_ig_localization/Qw...,complete,20,0.015097,0.038421,20,36,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
5,google_gemma-3-4b-pt,gemma_seed_stability,500,42,0.1,selected,animacy-circuit/results/eap_ig_localization/go...,animacy-circuit/results/eap_ig_localization/go...,animacy-circuit/results/eap_ig_localization/go...,complete,20,0.056377,0.110049,20,22,google_gemma-3-4b-pt / gemma_seed_stability / ...
8,gpt2,gpt2_seed_stability,500,42,0.1,selected,animacy-circuit/results/eap_ig_localization/gp...,animacy-circuit/results/eap_ig_localization/gp...,animacy-circuit/results/eap_ig_localization/gp...,complete,20,0.013647,0.018953,20,32,gpt2 / gpt2_seed_stability / s500 seed 42 / k=20
11,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,0.1,selected,animacy-circuit/results/eap_ig_localization/me...,animacy-circuit/results/eap_ig_localization/me...,animacy-circuit/results/eap_ig_localization/me...,complete,20,0.030803,0.045820,20,24,meta-llama_Llama-3.2-3B / llama_seed_stability...


,model_slug,run_name,sample_size,seed,selected_budget,selected_faithfulness_mean,selected_accuracy_mean,collapsed_edge,parent,child,signed_sum,abs_score,underlying_edges,rank,underlying_edge_count,slot
40,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,0.015097,0.038421,input->m0,input,m0,-0.004443,0.004443,input->m0,1,1,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
41,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,0.015097,0.038421,input->a0.h18,input,a0.h18,-0.003070,0.003070,input->a0.h18<q>|input->a0.h18<k>|input->a0.h1...,2,3,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
42,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,0.015097,0.038421,m28->logits,m28,logits,-0.002740,0.002740,m28->logits,3,1,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
43,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,0.015097,0.038421,m32->logits,m32,logits,-0.002615,0.002615,m32->logits,4,1,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
44,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,0.015097,0.038421,m0->m1,m0,m1,-0.002293,0.002293,m0->m1,5,1,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...


,model_slug,run_name,sample_size,seed,selected_budget,selected_faithfulness_mean,selected_accuracy_mean,threshold,collapsed_edge_index,underlying_edge_index,underlying_edge,collapsed_edge,collapsed_rank,parent,child,collapsed_abs_score,collapsed_signed_sum,collapsed_underlying_edge_count,slot
74,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,0.015097,0.038421,0.1,1,1,input->m0,input->m0,1,input,m0,0.004443,-0.004443,1,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
75,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,0.015097,0.038421,0.1,2,1,input->a0.h18<q>,input->a0.h18,2,input,a0.h18,0.003070,-0.003070,3,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
76,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,0.015097,0.038421,0.1,2,2,input->a0.h18<k>,input->a0.h18,2,input,a0.h18,0.003070,-0.003070,3,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
77,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,0.015097,0.038421,0.1,2,3,input->a0.h18<v>,input->a0.h18,2,input,a0.h18,0.003070,-0.003070,3,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
78,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,0.015097,0.038421,0.1,3,1,m28->logits,m28->logits,3,m28,logits,0.002740,-0.002740,1,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...


## Selected Budgets


In [18]:
from __future__ import annotations
budget_table = selected_summary[[
    "model_slug", "run_name", "sample_size", "seed", "selected_budget", "status"
] + [col for col in ["selected_faithfulness", "selected_accuracy", "selected_underlying_edge_count"] if col in selected_summary.columns]].sort_values(SLOT_COLUMNS)

budget_fig = px.bar(
    budget_table,
    x="model_slug",
    y="selected_budget",
    color="model_slug",
    hover_data=[col for col in ["run_name", "sample_size", "seed", "selected_faithfulness", "selected_accuracy", "selected_underlying_edge_count"] if col in budget_table.columns],
    title="Selected necessary collapsed-edge budgets",
)
budget_fig.update_layout(showlegend=False, xaxis_title="Model", yaxis_title="Selected budget")
show_fig(budget_fig, height=420)

display(budget_table)


,model_slug,run_name,sample_size,seed,selected_budget,status,selected_faithfulness,selected_accuracy,selected_underlying_edge_count
2,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,selected,0.015097,0.038421,36
5,google_gemma-3-4b-pt,gemma_seed_stability,500,42,20,selected,0.056377,0.110049,22
8,gpt2,gpt2_seed_stability,500,42,20,selected,0.013647,0.018953,32
11,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,selected,0.030803,0.045820,24


## Edge Functional Roles

These tables label each collapsed edge by parent/child component type and summarize whether each necessary subcircuit is dominated by input pickup, transfer, processing, or readout edges.


In [19]:
from __future__ import annotations
def node_metadata_frame(nodes: pd.Series) -> pd.DataFrame:
    rows = []
    for node in nodes.astype(str):
        meta = dict(eap_node_metadata(node))
        rows.append({"node": node, "kind": meta["kind"], "layer": meta["layer"], "head": meta["head"]})
    return pd.DataFrame(rows)


def role_for_edge(parent_type: str, child_type: str) -> str:
    if parent_type == "input" and child_type in {"attn", "mlp"}:
        return "input_to_component"
    if parent_type == "attn" and child_type == "mlp":
        return "head_to_mlp"
    if parent_type == "mlp" and child_type == "mlp":
        return "mlp_to_mlp"
    if child_type == "logits" and parent_type in {"attn", "mlp"}:
        return "component_to_logits"
    return "other"


def add_edge_semantics(edges: pd.DataFrame) -> pd.DataFrame:
    rows = edges.copy()
    parent_meta = node_metadata_frame(rows["parent"]).add_prefix("parent_")
    child_meta = node_metadata_frame(rows["child"]).add_prefix("child_")
    rows = pd.concat([rows.reset_index(drop=True), parent_meta, child_meta], axis=1)
    rows = rows.rename(
        columns={
            "parent_kind": "parent_type",
            "child_kind": "child_type",
            "parent_layer": "parent_layer",
            "child_layer": "child_layer",
        }
    )
    rows["edge_role"] = [
        role_for_edge(parent_type, child_type)
        for parent_type, child_type in zip(rows["parent_type"], rows["child_type"])
    ]
    rows["layer_pair"] = rows["parent_layer"].astype(str) + "->" + rows["child_layer"].astype(str)
    return rows


semantic_edges = add_edge_semantics(collapsed)
assert semantic_edges["edge_role"].notna().all()

edge_role_summary = (
    semantic_edges.groupby(SLOT_COLUMNS + ["selected_budget", "edge_role"], dropna=False)
    .agg(
        edge_count=("collapsed_edge", "count"),
        total_abs_score=("abs_score", "sum"),
        positive_signed_mass=("signed_sum", lambda values: values[values > 0].sum()),
        negative_signed_mass=("signed_sum", lambda values: values[values < 0].sum()),
        mean_rank=("rank", "mean"),
        top_examples=("collapsed_edge", lambda values: ", ".join(values.astype(str).head(5))),
    )
    .reset_index()
    .sort_values(SLOT_COLUMNS + ["total_abs_score"], ascending=[True, True, True, True, False], kind="stable")
)
edge_role_summary = edge_role_summary.assign(slot=slot_label(edge_role_summary))

layer_pair_summary = (
    semantic_edges.groupby(SLOT_COLUMNS + ["selected_budget", "edge_role", "layer_pair"], dropna=False)
    .agg(edge_count=("collapsed_edge", "count"), total_abs_score=("abs_score", "sum"), mean_rank=("rank", "mean"))
    .reset_index()
    .sort_values(SLOT_COLUMNS + ["total_abs_score"], ascending=[True, True, True, True, False], kind="stable")
)
layer_pair_summary = layer_pair_summary.assign(slot=slot_label(layer_pair_summary))

display(semantic_edges[[
    "slot", "collapsed_edge", "edge_role", "parent_type", "child_type",
    "parent_layer", "child_layer", "signed_sum", "abs_score", "rank", "underlying_edge_count"
]].head(20))
display(edge_role_summary)
display(layer_pair_summary.head(50))


,slot,collapsed_edge,edge_role,parent_type,child_type,parent_layer,child_layer,signed_sum,abs_score,rank,underlying_edge_count
0,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,input->m0,input_to_component,input,mlp,-1,0,-0.004443,0.004443,1,1
1,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,input->a0.h18,input_to_component,input,attn,-1,0,-0.003070,0.003070,2,3
2,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,m28->logits,component_to_logits,mlp,logits,28,-1,-0.002740,0.002740,3,1
3,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,m32->logits,component_to_logits,mlp,logits,32,-1,-0.002615,0.002615,4,1
4,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,m0->m1,mlp_to_mlp,mlp,mlp,0,1,-0.002293,0.002293,5,1
5,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,m31->logits,component_to_logits,mlp,logits,31,-1,-0.002219,0.002219,6,1
6,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,m0->m3,mlp_to_mlp,mlp,mlp,0,3,-0.002190,0.002190,7,1
7,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,input->a0.h15,input_to_component,input,attn,-1,0,-0.002137,0.002145,8,3
8,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,input->a0.h9,input_to_component,input,attn,-1,0,-0.001994,0.001994,9,3
9,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,a0.h18->m0,head_to_mlp,attn,mlp,0,0,-0.001837,0.001837,10,1


,model_slug,run_name,sample_size,seed,selected_budget,edge_role,edge_count,total_abs_score,positive_signed_mass,negative_signed_mass,mean_rank,top_examples,slot
2,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,input_to_component,9,0.019961,0.000000,-0.019884,10.555556,"input->m0, input->a0.h18, input->a0.h15, input...",Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
0,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,component_to_logits,6,0.012556,0.000000,-0.012556,9.833333,"m28->logits, m32->logits, m31->logits, m33->lo...",Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
3,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,mlp_to_mlp,4,0.007556,0.000000,-0.007556,11.500000,"m0->m1, m0->m3, m3->m5, m0->m5",Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
1,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,head_to_mlp,1,0.001837,0.000000,-0.001837,10.000000,a0.h18->m0,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
4,google_gemma-3-4b-pt,gemma_seed_stability,500,42,20,component_to_logits,11,0.014283,0.003894,-0.010389,10.545455,"m33->logits, m28->logits, m22->logits, m25->lo...",google_gemma-3-4b-pt / gemma_seed_stability / ...
6,google_gemma-3-4b-pt,gemma_seed_stability,500,42,20,input_to_component,6,0.010910,0.000000,-0.010809,7.666667,"input->m0, input->a3.h0, input->m5, input->m3,...",google_gemma-3-4b-pt / gemma_seed_stability / ...
7,google_gemma-3-4b-pt,gemma_seed_stability,500,42,20,mlp_to_mlp,2,0.001872,0.000000,-0.001872,15.000000,"m5->m6, m3->m4",google_gemma-3-4b-pt / gemma_seed_stability / ...
5,google_gemma-3-4b-pt,gemma_seed_stability,500,42,20,head_to_mlp,1,0.000895,0.000895,0.000000,18.000000,a0.h5->m0,google_gemma-3-4b-pt / gemma_seed_stability / ...
9,gpt2,gpt2_seed_stability,500,42,20,input_to_component,3,0.863247,0.000000,-0.863247,5.666667,"input->m0, input->a0.h4, input->a0.h3",gpt2 / gpt2_seed_stability / s500 seed 42 / k=20
8,gpt2,gpt2_seed_stability,500,42,20,component_to_logits,9,0.766579,0.050693,-0.715886,9.555556,"m10->logits, m9->logits, a9.h3->logits, m8->lo...",gpt2 / gpt2_seed_stability / s500 seed 42 / k=20


,model_slug,run_name,sample_size,seed,selected_budget,edge_role,layer_pair,edge_count,total_abs_score,mean_rank,slot
7,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,input_to_component,-1->0,9,0.019961,10.555556,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
1,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,component_to_logits,28->-1,1,0.002740,3.000000,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
4,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,component_to_logits,32->-1,1,0.002615,4.000000,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
8,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,mlp_to_mlp,0->1,1,0.002293,5.000000,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
3,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,component_to_logits,31->-1,1,0.002219,6.000000,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
9,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,mlp_to_mlp,0->3,1,0.002190,7.000000,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
6,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,head_to_mlp,0->0,1,0.001837,10.000000,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
5,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,component_to_logits,33->-1,1,0.001824,11.000000,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
2,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,component_to_logits,29->-1,1,0.001719,15.000000,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
11,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,mlp_to_mlp,3->5,1,0.001572,16.000000,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...


In [20]:
from __future__ import annotations
role_fig = px.bar(
    edge_role_summary,
    x="slot",
    y="total_abs_score",
    color="edge_role",
    hover_data=["edge_count", "positive_signed_mass", "negative_signed_mass", "top_examples"],
    title="Necessary-edge role distribution by absolute EAP score mass",
)
role_fig.update_layout(xaxis_tickangle=-35, xaxis_title="Slot", yaxis_title="Total absolute score")
show_fig(role_fig, height=620)


## Component Semantic Cards

Component cards aggregate incident necessary edges for each attention head or MLP. `input` and `logits` are structural endpoints and are excluded from this component table.


In [21]:
from __future__ import annotations
STRUCTURAL_NODES = {"input", "logits"}


def component_incident_edges(edges: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for edge in edges.to_dict("records"):
        for endpoint, direction in ((edge["parent"], "outgoing"), (edge["child"], "incoming")):
            meta = eap_node_metadata(str(endpoint))
            if meta["kind"] in {"input", "logits"}:
                continue
            rows.append(
                {
                    **{col: edge[col] for col in SLOT_COLUMNS + ["slot", "selected_budget"]},
                    "component": str(endpoint),
                    "component_type": meta["kind"],
                    "layer": int(meta["layer"]),
                    "head": meta["head"],
                    "direction": direction,
                    "edge_role": edge["edge_role"],
                    "collapsed_edge": edge["collapsed_edge"],
                    "rank": edge["rank"],
                    "signed_sum": edge["signed_sum"],
                    "abs_score": edge["abs_score"],
                }
            )
    return pd.DataFrame(rows)


def strongest_role_text(rows: pd.DataFrame, direction: str) -> str | None:
    subset = rows[rows["direction"] == direction]
    if subset.empty:
        return None
    grouped = (
        subset.groupby("edge_role")
        .agg(total_abs_score=("abs_score", "sum"), edge_count=("collapsed_edge", "count"))
        .reset_index()
        .sort_values(["total_abs_score", "edge_count"], ascending=[False, False], kind="stable")
    )
    top = grouped.iloc[0]
    return f"{top.edge_role} ({top.edge_count} edges, abs={top.total_abs_score:.4g})"


component_incidents = component_incident_edges(semantic_edges)
if component_incidents.empty:
    component_cards = pd.DataFrame()
else:
    component_cards = (
        component_incidents.groupby(SLOT_COLUMNS + ["slot", "selected_budget", "component", "component_type", "layer", "head"], dropna=False)
        .agg(
            incident_edge_count=("collapsed_edge", "nunique"),
            abs_incident_mass=("abs_score", "sum"),
            signed_incident_mass=("signed_sum", "sum"),
            best_rank=("rank", "min"),
            top_incident_edges=("collapsed_edge", lambda values: ", ".join(pd.Series(values).astype(str).drop_duplicates().head(5))),
        )
        .reset_index()
    )
    role_rows = []
    for key, group in component_incidents.groupby(SLOT_COLUMNS + ["component"], dropna=False):
        key_dict = dict(zip(SLOT_COLUMNS + ["component"], key))
        role_rows.append(
            {
                **key_dict,
                "strongest_incoming_role": strongest_role_text(group, "incoming"),
                "strongest_outgoing_role": strongest_role_text(group, "outgoing"),
            }
        )
    component_cards = component_cards.merge(pd.DataFrame(role_rows), on=SLOT_COLUMNS + ["component"], how="left")
    component_cards = component_cards.sort_values(SLOT_COLUMNS + ["best_rank", "component"], kind="stable")

if not component_cards.empty:
    component_cards["attention_target_summary"] = None
    component_cards["readout_direction"] = "activation_not_run"

assert component_cards.empty or component_cards["component"].notna().all()
display(component_cards)


,model_slug,run_name,sample_size,seed,slot,selected_budget,component,component_type,layer,head,incident_edge_count,abs_incident_mass,signed_incident_mass,best_rank,top_incident_edges,strongest_incoming_role,strongest_outgoing_role,attention_target_summary,readout_direction
8,Qwen_Qwen3-4B,qwen_seed_stability,500,42,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,20,m0,mlp,0,NaN,5,0.012265,-0.012265,1,"input->m0, m0->m1, m0->m3, a0.h18->m0, m0->m5","input_to_component (1 edges, abs=0.004443)","mlp_to_mlp (3 edges, abs=0.005984)",None,activation_not_run
1,Qwen_Qwen3-4B,qwen_seed_stability,500,42,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,20,a0.h18,attn,0,18.0,2,0.004907,-0.004907,2,"input->a0.h18, a0.h18->m0","input_to_component (1 edges, abs=0.00307)","head_to_mlp (1 edges, abs=0.001837)",None,activation_not_run
11,Qwen_Qwen3-4B,qwen_seed_stability,500,42,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,20,m28,mlp,28,NaN,1,0.002740,-0.002740,3,m28->logits,NaN,"component_to_logits (1 edges, abs=0.00274)",None,activation_not_run
15,Qwen_Qwen3-4B,qwen_seed_stability,500,42,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,20,m32,mlp,32,NaN,1,0.002615,-0.002615,4,m32->logits,NaN,"component_to_logits (1 edges, abs=0.002615)",None,activation_not_run
9,Qwen_Qwen3-4B,qwen_seed_stability,500,42,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...,20,m1,mlp,1,NaN,1,0.002293,-0.002293,5,m0->m1,"mlp_to_mlp (1 edges, abs=0.002293)",NaN,None,activation_not_run
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,meta-llama_Llama-3.2-3B / llama_seed_stability...,20,m20,mlp,20,NaN,1,0.012905,-0.012905,13,m20->logits,NaN,"component_to_logits (1 edges, abs=0.0129)",None,activation_not_run
60,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,meta-llama_Llama-3.2-3B / llama_seed_stability...,20,a3.h18,attn,3,18.0,1,0.011768,-0.008890,15,m2->a3.h18,"other (1 edges, abs=0.01177)",NaN,None,activation_not_run
68,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,meta-llama_Llama-3.2-3B / llama_seed_stability...,20,m25,mlp,25,NaN,1,0.011654,-0.011654,16,m25->logits,NaN,"component_to_logits (1 edges, abs=0.01165)",None,activation_not_run
66,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,meta-llama_Llama-3.2-3B / llama_seed_stability...,20,m21,mlp,21,NaN,1,0.009804,-0.009804,18,m21->logits,NaN,"component_to_logits (1 edges, abs=0.009804)",None,activation_not_run


In [22]:
from __future__ import annotations
component_type_summary = (
    component_cards.groupby(SLOT_COLUMNS + ["selected_budget", "component_type"], dropna=False)
    .agg(
        component_count=("component", "nunique"),
        total_abs_incident_mass=("abs_incident_mass", "sum"),
        mean_layer=("layer", "mean"),
        top_components=("component", lambda values: ", ".join(values.astype(str).head(6))),
    )
    .reset_index()
    .sort_values(SLOT_COLUMNS + ["total_abs_incident_mass"], ascending=[True, True, True, True, False], kind="stable")
)
component_type_summary = component_type_summary.assign(slot=slot_label(component_type_summary))

layer_distribution = (
    component_cards.groupby(SLOT_COLUMNS + ["selected_budget", "component_type", "layer"], dropna=False)
    .agg(component_count=("component", "nunique"), total_abs_incident_mass=("abs_incident_mass", "sum"))
    .reset_index()
    .sort_values(SLOT_COLUMNS + ["layer", "component_type"], kind="stable")
)
layer_distribution = layer_distribution.assign(slot=slot_label(layer_distribution))

display(component_type_summary)
display(layer_distribution.head(80))

component_fig = px.bar(
    component_type_summary,
    x="slot",
    y="component_count",
    color="component_type",
    barmode="group",
    hover_data=["top_components", "total_abs_incident_mass", "mean_layer"],
    title="Necessary component participation",
)
component_fig.update_layout(xaxis_tickangle=-35, xaxis_title="Slot", yaxis_title="Component count")
show_fig(component_fig, height=620)


,model_slug,run_name,sample_size,seed,selected_budget,component_type,component_count,total_abs_incident_mass,mean_layer,top_components,slot
1,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,mlp,10,0.033949,18.500000,"m0, m28, m32, m1, m31, m3",Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
0,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,attn,8,0.017354,0.000000,"a0.h18, a0.h15, a0.h9, a0.h22, a0.h5, a0.h21",Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
3,google_gemma-3-4b-pt,gemma_seed_stability,500,42,20,mlp,17,0.028137,18.058824,"m0, m33, m28, m22, m25, m23",google_gemma-3-4b-pt / gemma_seed_stability / ...
2,google_gemma-3-4b-pt,gemma_seed_stability,500,42,20,attn,2,0.002589,1.500000,"a3.h0, a0.h5",google_gemma-3-4b-pt / gemma_seed_stability / ...
5,gpt2,gpt2_seed_stability,500,42,20,mlp,10,1.916787,5.700000,"m0, m1, m10, m9, m8, m3",gpt2 / gpt2_seed_stability / s500 seed 42 / k=20
4,gpt2,gpt2_seed_stability,500,42,20,attn,9,0.664406,5.000000,"a9.h3, a0.h4, a8.h7, a0.h3, a5.h7, a10.h11",gpt2 / gpt2_seed_stability / s500 seed 42 / k=20
7,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,mlp,11,0.500032,15.090909,"m0, m27, m1, m2, m3, m19",meta-llama_Llama-3.2-3B / llama_seed_stability...
6,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,attn,5,0.073724,16.600000,"a27.h0, a1.h3, a27.h13, a3.h18, a25.h23",meta-llama_Llama-3.2-3B / llama_seed_stability...


,model_slug,run_name,sample_size,seed,selected_budget,component_type,layer,component_count,total_abs_incident_mass,slot
0,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,attn,0,8,0.017354,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
1,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,mlp,0,1,0.012265,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
2,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,mlp,1,1,0.002293,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
3,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,mlp,3,1,0.003762,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
4,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,mlp,5,1,0.003073,Qwen_Qwen3-4B / qwen_seed_stability / s500 see...
...,...,...,...,...,...,...,...,...,...,...
49,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,attn,25,1,0.008889,meta-llama_Llama-3.2-3B / llama_seed_stability...
59,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,mlp,25,1,0.011654,meta-llama_Llama-3.2-3B / llama_seed_stability...
60,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,mlp,26,1,0.015502,meta-llama_Llama-3.2-3B / llama_seed_stability...
50,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,attn,27,2,0.040026,meta-llama_Llama-3.2-3B / llama_seed_stability...


## Hook Mapping and Run Metadata

These helpers map collapsed EAP nodes to TransformerLens hook names and recover model/target metadata from each localization slot. Activation cells use the same repo defaults as localization when older summaries do not record target settings.


In [23]:
from __future__ import annotations
MODEL_SLUG_OVERRIDES = {
    "gpt2": "gpt2",
    "Qwen_Qwen3-4B": "Qwen/Qwen3-4B",
    "google_gemma-3-4b-pt": "google/gemma-3-4b-pt",
    "meta-llama_Llama-3.2-3B": "meta-llama/Llama-3.2-3B",
}


def model_name_from_slug(model_slug: str) -> str:
    if model_slug in MODEL_SLUG_OVERRIDES:
        return MODEL_SLUG_OVERRIDES[model_slug]
    if "_" in model_slug:
        return model_slug.replace("_", "/", 1)
    return model_slug


def resolve_artifact_path(value: str | Path | float | None) -> Path | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    path = Path(str(value))
    if path.is_absolute():
        return path
    for root in (PROJECT_ROOT.parent, PROJECT_ROOT, Path.cwd().resolve()):
        candidate = root / path
        if candidate.exists():
            return candidate
    return PROJECT_ROOT.parent / path


def localization_summary_path(slot_row: pd.Series | dict[str, Any]) -> Path | None:
    edge_path = resolve_artifact_path(slot_row.get("edge_path")) if hasattr(slot_row, "get") else None
    if edge_path is None or not edge_path.exists():
        return None
    sample_size = int(slot_row["sample_size"])
    seed = int(slot_row["seed"])
    candidate = edge_path.with_name(f"localization_summary_sample_{sample_size}_seed_{seed}.json")
    return candidate if candidate.exists() else None


def slot_run_metadata(slot_row: pd.Series | dict[str, Any]) -> dict[str, Any]:
    model_slug = str(slot_row["model_slug"])
    metadata = {
        "model_slug": model_slug,
        "model_name": model_name_from_slug(model_slug),
        "target_source": None,
        "target_filter_policy": "model_success",
        "target_settings_source": "repo_default",
        "summary_path": None,
    }
    path = localization_summary_path(slot_row)
    if path is None:
        return metadata
    payload = json.loads(path.read_text(encoding="utf-8"))
    config = payload.get("config", {}) or {}
    dataset_summary = payload.get("dataset_summary", {}) or {}
    metadata.update(
        {
            "model_name": dataset_summary.get("target_model") or config.get("model_name") or metadata["model_name"],
            "target_source": config.get("target_source") or payload.get("target_source"),
            "target_filter_policy": config.get("target_filter_policy", metadata["target_filter_policy"]),
            "summary_path": str(path),
        }
    )
    if metadata["target_source"] is not None:
        metadata["target_settings_source"] = "localization_summary"
    return metadata


def component_hook_spec(component: str) -> dict[str, Any]:
    meta = eap_node_metadata(component)
    if meta["kind"] == "mlp":
        return {
            "component": component,
            "component_type": "mlp",
            "layer": int(meta["layer"]),
            "head": None,
            "activation_hook": f"blocks.{int(meta['layer'])}.hook_mlp_out",
            "pattern_hook": None,
        }
    if meta["kind"] == "attn":
        return {
            "component": component,
            "component_type": "attn",
            "layer": int(meta["layer"]),
            "head": int(meta["head"]),
            "activation_hook": f"blocks.{int(meta['layer'])}.attn.hook_result",
            "pattern_hook": f"blocks.{int(meta['layer'])}.attn.hook_pattern",
        }
    return {
        "component": component,
        "component_type": meta["kind"],
        "layer": meta["layer"],
        "head": meta["head"],
        "activation_hook": None,
        "pattern_hook": None,
    }


hook_specs = pd.DataFrame([component_hook_spec(component) for component in sorted(component_cards["component"].dropna().unique())])
display(hook_specs.head(50))

def hooks_for_components(components: list[str], include_patterns: bool = True) -> list[str]:
    hooks = []
    for component in components:
        spec = component_hook_spec(component)
        if spec["activation_hook"]:
            hooks.append(spec["activation_hook"])
        if include_patterns and spec["pattern_hook"]:
            hooks.append(spec["pattern_hook"])
    return sorted(set(hooks))


,component,component_type,layer,head,activation_hook,pattern_hook
0,a0.h15,attn,0,15.0,blocks.0.attn.hook_result,blocks.0.attn.hook_pattern
1,a0.h18,attn,0,18.0,blocks.0.attn.hook_result,blocks.0.attn.hook_pattern
2,a0.h2,attn,0,2.0,blocks.0.attn.hook_result,blocks.0.attn.hook_pattern
3,a0.h21,attn,0,21.0,blocks.0.attn.hook_result,blocks.0.attn.hook_pattern
4,a0.h22,attn,0,22.0,blocks.0.attn.hook_result,blocks.0.attn.hook_pattern
5,a0.h3,attn,0,3.0,blocks.0.attn.hook_result,blocks.0.attn.hook_pattern
6,a0.h4,attn,0,4.0,blocks.0.attn.hook_result,blocks.0.attn.hook_pattern
7,a0.h5,attn,0,5.0,blocks.0.attn.hook_result,blocks.0.attn.hook_pattern
8,a0.h8,attn,0,8.0,blocks.0.attn.hook_result,blocks.0.attn.hook_pattern
9,a0.h9,attn,0,9.0,blocks.0.attn.hook_result,blocks.0.attn.hook_pattern


## Activation Semantics Helpers

Run these functions on a GPU-capable environment. By default, the notebook prepares callable functions but does not load every model automatically.


In [24]:
from __future__ import annotations
def normalize_prompt_columns(df: pd.DataFrame) -> pd.DataFrame:
    rows = df.copy()
    if "clean_prefix" not in rows.columns and "clean" in rows.columns:
        rows["clean_prefix"] = rows["clean"]
    if "corrupt_prefix" not in rows.columns and "corrupt" in rows.columns:
        rows["corrupt_prefix"] = rows["corrupt"]
    missing = sorted({"clean_prefix", "corrupt_prefix"} - set(rows.columns))
    if missing:
        raise ValueError(f"Dataset is missing prompt columns: {missing}")
    return rows


def sample_examples(df: pd.DataFrame, n_examples: int, seed: int) -> pd.DataFrame:
    if n_examples is None or n_examples >= len(df):
        return df.reset_index(drop=True).copy()
    return df.sample(n=n_examples, random_state=seed).reset_index(drop=True).copy()


def load_slot_examples(slot_row: pd.Series, n_examples: int = N_EXAMPLES, prompt_pairs: pd.DataFrame | None = None) -> tuple[pd.DataFrame, dict[str, Any]]:
    from circuit_finder_core import add_sequence_lengths, filter_df_to_prompt_pairs, load_policy_filtered_dataset, load_model_context

    metadata = slot_run_metadata(slot_row)
    context = load_model_context(PROJECT_ROOT, metadata["model_name"], target_source=metadata["target_source"])
    model = context["model"]
    raw_df = load_policy_filtered_dataset(
        project_root=PROJECT_ROOT,
        model_name=metadata["model_name"],
        batch_size=BATCH_SIZE,
        target_filter_policy=metadata["target_filter_policy"],
        cache=True,
        max_examples=None,
        seed=int(slot_row["seed"]),
        target_source=metadata["target_source"],
    )
    raw_df = normalize_prompt_columns(raw_df)
    if prompt_pairs is not None and not prompt_pairs.empty:
        raw_df = filter_df_to_prompt_pairs(raw_df, prompt_pairs)
    sampled = sample_examples(raw_df, n_examples=n_examples, seed=int(slot_row["seed"]))
    sampled = add_sequence_lengths(sampled, model)
    if sampled.empty:
        raise ValueError("No same-length clean/corrupt examples survived tokenization filtering for this slot.")
    return sampled, {**context, **metadata}


def single_token_position(tokenizer, text: str, token_text: str | None) -> int | None:
    from circuit_finder_core import component_token_span

    if token_text is None or pd.isna(token_text):
        return None
    span, error = component_token_span(tokenizer, str(text), str(token_text))
    if span is None or error is not None:
        return None
    if span[1] - span[0] != 1:
        return None
    return int(span[0] + 1)  # +1 for prepended BOS.


def add_key_positions(df: pd.DataFrame, tokenizer) -> pd.DataFrame:
    rows = df.reset_index(drop=True).copy()
    clean_patient_positions = []
    corrupt_patient_positions = []
    clean_verb_positions = []
    corrupt_verb_positions = []
    clean_by_positions = []
    corrupt_by_positions = []
    for row in rows.to_dict("records"):
        patient = row.get("patient", row.get("patient_x", row.get("patient_y")))
        clean_verb = row.get("clean_verb", row.get("clean_verb_x", row.get("clean_verb_y")))
        corrupt_verb = row.get("corrupt_verb", row.get("corrupt_verb_x", row.get("corrupt_verb_y")))
        clean_patient_positions.append(single_token_position(tokenizer, row["clean_prefix"], patient))
        corrupt_patient_positions.append(single_token_position(tokenizer, row["corrupt_prefix"], patient))
        clean_verb_positions.append(single_token_position(tokenizer, row["clean_prefix"], clean_verb))
        corrupt_verb_positions.append(single_token_position(tokenizer, row["corrupt_prefix"], corrupt_verb))
        clean_by_positions.append(single_token_position(tokenizer, row["clean_prefix"], "by"))
        corrupt_by_positions.append(single_token_position(tokenizer, row["corrupt_prefix"], "by"))
    rows["bos_pos"] = 0
    rows["final_pos"] = rows["seq_len"].astype(int) - 1
    rows["clean_patient_pos"] = clean_patient_positions
    rows["corrupt_patient_pos"] = corrupt_patient_positions
    rows["clean_verb_pos"] = clean_verb_positions
    rows["corrupt_verb_pos"] = corrupt_verb_positions
    rows["clean_by_pos"] = clean_by_positions
    rows["corrupt_by_pos"] = corrupt_by_positions
    return rows


def unembedding_matrix(model):
    if hasattr(model, "W_U"):
        return model.W_U
    if hasattr(model, "unembed") and hasattr(model.unembed, "W_U"):
        return model.unembed.W_U
    raise AttributeError("Could not locate model unembedding matrix W_U.")


def animacy_readout_direction(model, animate_ids_tensor, inanimate_ids_tensor):
    import torch

    w_u = unembedding_matrix(model)
    animate_ids = animate_ids_tensor.to(device=w_u.device, dtype=torch.long)
    inanimate_ids = inanimate_ids_tensor.to(device=w_u.device, dtype=torch.long)
    return w_u[:, animate_ids].mean(dim=1) - w_u[:, inanimate_ids].mean(dim=1)


def nearest_centroid_accuracy(features: np.ndarray, labels: np.ndarray) -> float | None:
    unique = sorted(set(labels.tolist()))
    if len(unique) < 2:
        return None
    centroids = {label: features[labels == label].mean(axis=0) for label in unique}
    correct = 0
    for vector, label in zip(features, labels):
        predicted = min(unique, key=lambda candidate: float(np.linalg.norm(vector - centroids[candidate])))
        correct += int(predicted == label)
    return float(correct / len(labels)) if len(labels) else None


def pca_geometry_summary(activation_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    try:
        from sklearn.decomposition import PCA
    except Exception as exc:
        return pd.DataFrame(), pd.DataFrame([{"status": "skipped", "reason": f"sklearn unavailable: {exc}"}])

    summaries = []
    projections = []
    group_cols = SLOT_COLUMNS + ["component", "component_type", "layer", "head", "position_label"]
    for key, group in activation_df.groupby(group_cols, dropna=False):
        clean = np.stack(group[group["condition"] == "clean"]["activation"].to_numpy()) if (group["condition"] == "clean").any() else np.empty((0, 0))
        corrupt = np.stack(group[group["condition"] == "corrupt"]["activation"].to_numpy()) if (group["condition"] == "corrupt").any() else np.empty((0, 0))
        key_dict = dict(zip(group_cols, key))
        if clean.shape[0] < 2 or corrupt.shape[0] < 2:
            summaries.append({**key_dict, "status": "skipped", "reason": "fewer than two examples per condition"})
            continue
        clean_map = {row["example_id"]: row["activation"] for row in group[group["condition"] == "clean"].to_dict("records")}
        corrupt_map = {row["example_id"]: row["activation"] for row in group[group["condition"] == "corrupt"].to_dict("records")}
        paired_ids = sorted(set(clean_map).intersection(corrupt_map))
        delta_norms = np.array([float(np.linalg.norm(clean_map[item] - corrupt_map[item])) for item in paired_ids], dtype=float)
        features = np.concatenate([clean, corrupt], axis=0)
        labels = np.array(["clean"] * len(clean) + ["corrupt"] * len(corrupt))
        if features.shape[0] < 3 or features.shape[1] < 2:
            summaries.append({**key_dict, "status": "skipped", "reason": "insufficient PCA dimensions"})
            continue
        pca = PCA(n_components=2, random_state=0)
        xy = pca.fit_transform(features)
        clean_centroid = xy[labels == "clean"].mean(axis=0)
        corrupt_centroid = xy[labels == "corrupt"].mean(axis=0)
        summaries.append(
            {
                **key_dict,
                "status": "ok",
                "example_count": int(features.shape[0]),
                "pca_explained_variance_1": float(pca.explained_variance_ratio_[0]),
                "pca_explained_variance_2": float(pca.explained_variance_ratio_[1]),
                "clean_corrupt_centroid_distance": float(np.linalg.norm(clean_centroid - corrupt_centroid)),
                "nearest_centroid_accuracy": nearest_centroid_accuracy(xy, labels),
                "paired_delta_count": int(len(delta_norms)),
                "delta_vector_norm_mean": float(delta_norms.mean()) if len(delta_norms) else None,
                "delta_vector_norm_std": float(delta_norms.std()) if len(delta_norms) else None,
            }
        )
        for idx, (x, y) in enumerate(xy):
            projections.append({**key_dict, "condition": labels[idx], "pc1": float(x), "pc2": float(y)})
    return pd.DataFrame(summaries), pd.DataFrame(projections)


In [25]:
from __future__ import annotations
def cache_component_activation(cache, spec: dict[str, Any], batch_index: int, position: int):
    value = cache[spec["activation_hook"]]
    if spec["component_type"] == "attn":
        if value.ndim != 4:
            raise ValueError(f"Expected attention result cache with 4 dims for {spec['component']}, got {tuple(value.shape)}")
        return value[batch_index, position, int(spec["head"]), :]
    if value.ndim != 3:
        raise ValueError(f"Expected MLP cache with 3 dims for {spec['component']}, got {tuple(value.shape)}")
    return value[batch_index, position, :]


def append_readout_and_activation_rows(
    *,
    rows: list[dict[str, Any]],
    activations: list[dict[str, Any]],
    slot_row: pd.Series,
    component: str,
    spec: dict[str, Any],
    condition: str,
    position_label: str,
    batch_index: int,
    example_id: Any,
    position: int | None,
    cache,
    readout_dir,
):
    if position is None or pd.isna(position):
        return
    position = int(position)
    vector = cache_component_activation(cache, spec, batch_index, position).detach().float().cpu()
    readout = float(vector.to(readout_dir.device).dot(readout_dir).detach().float().cpu().item())
    base = {col: slot_row[col] for col in SLOT_COLUMNS}
    base.update(
        {
            "selected_budget": int(slot_row["selected_budget"]),
            "component": component,
            "component_type": spec["component_type"],
            "layer": spec["layer"],
            "head": spec["head"],
            "condition": condition,
            "example_id": example_id,
            "position_label": position_label,
            "position": position,
        }
    )
    rows.append({**base, "readout": readout})
    activations.append({**base, "activation": vector.numpy()})


def append_attention_rows(
    *,
    rows: list[dict[str, Any]],
    slot_row: pd.Series,
    component: str,
    spec: dict[str, Any],
    condition: str,
    batch_index: int,
    query_position: int,
    key_positions: dict[str, int | None],
    cache,
):
    pattern_hook = spec.get("pattern_hook")
    if not pattern_hook or pattern_hook not in cache:
        return
    pattern = cache[pattern_hook]
    if pattern.ndim != 4:
        return
    head_pattern = pattern[batch_index, int(spec["head"]), int(query_position), :].detach().float().cpu()
    base = {col: slot_row[col] for col in SLOT_COLUMNS}
    base.update(
        {
            "selected_budget": int(slot_row["selected_budget"]),
            "component": component,
            "component_type": spec["component_type"],
            "layer": spec["layer"],
            "head": spec["head"],
            "condition": condition,
            "query_position_label": "final_token",
            "query_position": int(query_position),
        }
    )
    for label, position in key_positions.items():
        if position is None or pd.isna(position):
            continue
        position = int(position)
        if 0 <= position < head_pattern.shape[0]:
            rows.append({**base, "key_position_label": label, "key_position": position, "attention_mass": float(head_pattern[position].item())})


def analyze_slot_activations(slot_row: pd.Series, n_examples: int = N_EXAMPLES, batch_size: int = BATCH_SIZE, prompt_pairs: pd.DataFrame | None = None) -> dict[str, pd.DataFrame]:
    import torch
    from circuit_finder_core import generate_exact_length_batches

    slot_edges = semantic_edges.merge(pd.DataFrame([slot_row[SLOT_COLUMNS].to_dict()]), on=SLOT_COLUMNS, how="inner")
    slot_components = component_cards.merge(pd.DataFrame([slot_row[SLOT_COLUMNS].to_dict()]), on=SLOT_COLUMNS, how="inner")
    components = slot_components["component"].dropna().astype(str).drop_duplicates().tolist()
    specs = {component: component_hook_spec(component) for component in components}
    supported_components = [component for component, spec in specs.items() if spec["activation_hook"] is not None]
    if not supported_components:
        return {"attention_semantics": pd.DataFrame(), "readout_semantics": pd.DataFrame(), "activation_geometry_summary": pd.DataFrame(), "activation_pca_projection": pd.DataFrame()}

    examples, context = load_slot_examples(slot_row, n_examples=n_examples, prompt_pairs=prompt_pairs)
    model = context["model"]
    tokenizer = context["tokenizer"]
    examples = add_key_positions(examples, tokenizer)
    readout_dir = animacy_readout_direction(model, context["animate_ids_tensor"], context["inanimate_ids_tensor"])
    hook_names = hooks_for_components(supported_components, include_patterns=True)
    available_hooks = set(name for name, _hook in model.hook_dict.items()) if hasattr(model, "hook_dict") else set(hook_names)
    hook_names = [hook for hook in hook_names if hook in available_hooks]
    unsupported = sorted(set(hooks_for_components(supported_components, include_patterns=True)) - set(hook_names))
    if unsupported:
        print(f"Skipping unsupported hooks for {slot_row['model_slug']} seed {slot_row['seed']}: {unsupported}")

    readout_rows: list[dict[str, Any]] = []
    activation_rows: list[dict[str, Any]] = []
    attention_rows: list[dict[str, Any]] = []
    device = model.cfg.device

    for clean_tokens, corrupt_tokens, batch_df in generate_exact_length_batches(examples, model, batch_size=batch_size, device=device):
        with torch.no_grad():
            _, clean_cache = model.run_with_cache(clean_tokens, names_filter=hook_names)
            _, corrupt_cache = model.run_with_cache(corrupt_tokens, names_filter=hook_names)
        for batch_index, (example_id, row) in enumerate(batch_df.iterrows()):
            position_map = {
                "final_token": (row["final_pos"], row["final_pos"]),
                "verb_token": (row.get("clean_verb_pos"), row.get("corrupt_verb_pos")),
                "patient_token": (row.get("clean_patient_pos"), row.get("corrupt_patient_pos")),
                "by_token": (row.get("clean_by_pos"), row.get("corrupt_by_pos")),
            }
            for component in supported_components:
                spec = specs[component]
                if spec["activation_hook"] not in clean_cache or spec["activation_hook"] not in corrupt_cache:
                    continue
                for position_label, (clean_pos, corrupt_pos) in position_map.items():
                    append_readout_and_activation_rows(
                        rows=readout_rows,
                        activations=activation_rows,
                        slot_row=slot_row,
                        component=component,
                        spec=spec,
                        condition="clean",
                        position_label=position_label,
                        batch_index=batch_index,
                        example_id=example_id,
                        position=clean_pos,
                        cache=clean_cache,
                        readout_dir=readout_dir,
                    )
                    append_readout_and_activation_rows(
                        rows=readout_rows,
                        activations=activation_rows,
                        slot_row=slot_row,
                        component=component,
                        spec=spec,
                        condition="corrupt",
                        position_label=position_label,
                        batch_index=batch_index,
                        example_id=example_id,
                        position=corrupt_pos,
                        cache=corrupt_cache,
                        readout_dir=readout_dir,
                    )
                if spec["component_type"] == "attn" and spec.get("pattern_hook") in clean_cache:
                    append_attention_rows(
                        rows=attention_rows,
                        slot_row=slot_row,
                        component=component,
                        spec=spec,
                        condition="clean",
                        batch_index=batch_index,
                        query_position=int(row["final_pos"]),
                        key_positions={
                            "BOS": row["bos_pos"],
                            "patient": row.get("clean_patient_pos"),
                            "verb": row.get("clean_verb_pos"),
                            "by": row.get("clean_by_pos"),
                            "final": row["final_pos"],
                        },
                        cache=clean_cache,
                    )
                    append_attention_rows(
                        rows=attention_rows,
                        slot_row=slot_row,
                        component=component,
                        spec=spec,
                        condition="corrupt",
                        batch_index=batch_index,
                        query_position=int(row["final_pos"]),
                        key_positions={
                            "BOS": row["bos_pos"],
                            "patient": row.get("corrupt_patient_pos"),
                            "verb": row.get("corrupt_verb_pos"),
                            "by": row.get("corrupt_by_pos"),
                            "final": row["final_pos"],
                        },
                        cache=corrupt_cache,
                    )
        del clean_cache, corrupt_cache

    readout_df = pd.DataFrame(readout_rows)
    activation_df = pd.DataFrame(activation_rows)
    attention_df = pd.DataFrame(attention_rows)

    if not readout_df.empty:
        readout_summary = (
            readout_df.groupby(SLOT_COLUMNS + ["selected_budget", "component", "component_type", "layer", "head", "position_label", "condition"], dropna=False)
            .agg(readout_mean=("readout", "mean"), readout_std=("readout", "std"), example_count=("readout", "count"))
            .reset_index()
        )
        pivot = readout_summary.pivot_table(
            index=SLOT_COLUMNS + ["selected_budget", "component", "component_type", "layer", "head", "position_label"],
            columns="condition",
            values="readout_mean",
            aggfunc="first",
        ).reset_index()
        if {"clean", "corrupt"}.issubset(pivot.columns):
            pivot["clean_minus_corrupt_readout"] = pivot["clean"] - pivot["corrupt"]

        def classify_readout_row(row, eps: float = 1e-6) -> str:
            values = [row.get("clean"), row.get("corrupt")]
            values = [float(value) for value in values if pd.notna(value)]
            if not values or max(abs(value) for value in values) <= eps:
                return "low_readout"
            signs = {1 if value > eps else -1 if value < -eps else 0 for value in values}
            if signs == {1}:
                return "animate_pushing"
            if signs == {-1}:
                return "inanimate_pushing"
            return "mixed"

        pivot["readout_direction"] = pivot.apply(classify_readout_row, axis=1)
        readout_summary = readout_summary.merge(pivot, on=SLOT_COLUMNS + ["selected_budget", "component", "component_type", "layer", "head", "position_label"], how="left")
    else:
        readout_summary = pd.DataFrame()

    if not attention_df.empty:
        attention_summary = (
            attention_df.groupby(SLOT_COLUMNS + ["selected_budget", "component", "layer", "head", "condition", "query_position_label", "key_position_label"], dropna=False)
            .agg(attention_mass_mean=("attention_mass", "mean"), attention_mass_std=("attention_mass", "std"), example_count=("attention_mass", "count"))
            .reset_index()
        )
    else:
        attention_summary = pd.DataFrame()

    geometry_summary, pca_projection = pca_geometry_summary(activation_df) if not activation_df.empty else (pd.DataFrame(), pd.DataFrame())

    del context["model"]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "examples": examples,
        "attention_semantics": attention_summary,
        "readout_semantics": readout_summary,
        "activation_geometry_summary": geometry_summary,
        "activation_pca_projection": pca_projection,
        "target_metadata": pd.DataFrame([{k: v for k, v in context.items() if k not in {"tokenizer", "animate_ids_tensor", "inanimate_ids_tensor"}}]),
    }


## Optional Activation Run

By default, this section loads precomputed GPU results from the latest `necessary_semantics_activations_*` directory. Run `scripts/executable/run_necessary_semantic_activations.py` on a GPU node to create that bundle. Set `LOAD_ACTIVATION_RESULTS = False` and `RUN_ACTIVATION_ANALYSIS = True` only for an in-notebook GPU run.


In [26]:
from __future__ import annotations
RUN_ACTIVATION_ANALYSIS = False
USE_PROMPT_INTERSECTION = True

def latest_activation_results_dir() -> Path | None:
    candidates = sorted(RESULTS_ROOT.glob("necessary_semantics_activations_*"))
    return candidates[-1] if candidates else None


def read_optional_csv(directory: Path, name: str) -> pd.DataFrame:
    path = directory / f"{name}.csv"
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


loaded_activation_results_dir = None
attention_semantics = pd.DataFrame()
readout_semantics = pd.DataFrame()
activation_geometry_summary = pd.DataFrame()
activation_pca_projection = pd.DataFrame()
target_metadata = pd.DataFrame()

if LOAD_ACTIVATION_RESULTS:
    candidate_dir = resolve_existing_path(ACTIVATION_RESULTS_DIR) if ACTIVATION_RESULTS_DIR is not None else latest_activation_results_dir()
    if candidate_dir is None or not candidate_dir.exists():
        print("No precomputed activation results found. Run scripts/executable/run_necessary_semantic_activations.py on a GPU node first.")
    else:
        loaded_activation_results_dir = candidate_dir
        attention_semantics = read_optional_csv(candidate_dir, "attention_semantics")
        readout_semantics = read_optional_csv(candidate_dir, "readout_semantics")
        activation_geometry_summary = read_optional_csv(candidate_dir, "activation_geometry_summary")
        activation_pca_projection = read_optional_csv(candidate_dir, "activation_pca_projection")
        target_metadata = read_optional_csv(candidate_dir, "target_metadata")
        print(f"Loaded precomputed activation results from {candidate_dir}")
        print(
            "Loaded rows:",
            f"attention={len(attention_semantics)}",
            f"readout={len(readout_semantics)}",
            f"geometry={len(activation_geometry_summary)}",
            f"pca={len(activation_pca_projection)}",
        )
        if attention_semantics.empty or readout_semantics.empty or activation_geometry_summary.empty or activation_pca_projection.empty:
            raise ValueError(f"Activation result CSVs were found in {candidate_dir}, but one or more loaded tables are empty.")

activation_results: list[dict[str, pd.DataFrame]] = []
if RUN_ACTIVATION_ANALYSIS and not LOAD_ACTIVATION_RESULTS:
    prompt_pairs = None
    if USE_PROMPT_INTERSECTION:
        # Intersection loading can still touch model-success caches, so it is kept inside the opt-in block.
        from circuit_finder_core import intersect_prompt_pair_frames, load_policy_filtered_dataset, prompt_pair_columns

        prompt_frames = []
        for _, slot_row in selected_summary.drop_duplicates("model_slug").iterrows():
            meta = slot_run_metadata(slot_row)
            try:
                frame = normalize_prompt_columns(
                    load_policy_filtered_dataset(
                        project_root=PROJECT_ROOT,
                        model_name=meta["model_name"],
                        batch_size=BATCH_SIZE,
                        target_filter_policy=meta["target_filter_policy"],
                        cache=True,
                        max_examples=None,
                        seed=int(slot_row["seed"]),
                        target_source=meta["target_source"],
                    )
                )
                prompt_frames.append(prompt_pair_columns(frame))
            except Exception as exc:
                print(f"Could not load prompt pairs for {slot_row['model_slug']}: {exc}")
        if prompt_frames:
            prompt_pairs = intersect_prompt_pair_frames(prompt_frames)
            print(f"Using {len(prompt_pairs)} prompt pairs in the cross-model intersection.")

    for _, slot_row in selected_summary.sort_values(SLOT_COLUMNS).iterrows():
        print(f"Analyzing {slot_row['slot']} with n_examples={N_EXAMPLES}")
        activation_results.append(analyze_slot_activations(slot_row, n_examples=N_EXAMPLES, batch_size=BATCH_SIZE, prompt_pairs=prompt_pairs))


def concat_result_table(results: list[dict[str, pd.DataFrame]], key: str) -> pd.DataFrame:
    frames = [result[key] for result in results if key in result and isinstance(result[key], pd.DataFrame) and not result[key].empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if activation_results:
    attention_semantics = concat_result_table(activation_results, "attention_semantics")
    readout_semantics = concat_result_table(activation_results, "readout_semantics")
    activation_geometry_summary = concat_result_table(activation_results, "activation_geometry_summary")
    activation_pca_projection = concat_result_table(activation_results, "activation_pca_projection")
    target_metadata = concat_result_table(activation_results, "target_metadata")

display(attention_semantics.head() if not attention_semantics.empty else "Activation analysis not run yet.")
display(readout_semantics.head() if not readout_semantics.empty else "Readout semantics not available yet.")
display(activation_geometry_summary.head() if not activation_geometry_summary.empty else "Activation geometry not available yet.")


Loaded precomputed activation results from /gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/eap_ig_localization/necessary_semantics_activations_2026-06-21
Loaded rows: attention=288 readout=720 geometry=360 pca=360000


,model_slug,run_name,sample_size,seed,selected_budget,component,layer,head,condition,query_position_label,key_position_label,attention_mass_mean,attention_mass_std,example_count
0,gpt2,gpt2_seed_stability,500,42,20,a0.h3,0,3,clean,final_token,BOS,0.043744,0.001241,500
1,gpt2,gpt2_seed_stability,500,42,20,a0.h3,0,3,clean,final_token,by,0.167632,0.004755,500
2,gpt2,gpt2_seed_stability,500,42,20,a0.h3,0,3,clean,final_token,final,0.609896,0.017299,500
3,gpt2,gpt2_seed_stability,500,42,20,a0.h3,0,3,clean,final_token,patient,0.020907,0.007149,500
4,gpt2,gpt2_seed_stability,500,42,20,a0.h3,0,3,clean,final_token,verb,0.085060,0.025339,500


,model_slug,run_name,sample_size,seed,selected_budget,component,component_type,layer,head,position_label,condition,readout_mean,readout_std,example_count,clean,corrupt,clean_minus_corrupt_readout,readout_direction
0,gpt2,gpt2_seed_stability,500,42,20,a0.h3,attn,0,3.0,by_token,clean,0.462294,0.094787,500,0.462294,0.276792,0.185502,animate_pushing
1,gpt2,gpt2_seed_stability,500,42,20,a0.h3,attn,0,3.0,by_token,corrupt,0.276792,0.134329,500,0.462294,0.276792,0.185502,animate_pushing
2,gpt2,gpt2_seed_stability,500,42,20,a0.h3,attn,0,3.0,final_token,clean,0.330143,0.022792,500,0.330143,0.296429,0.033714,animate_pushing
3,gpt2,gpt2_seed_stability,500,42,20,a0.h3,attn,0,3.0,final_token,corrupt,0.296429,0.033069,500,0.330143,0.296429,0.033714,animate_pushing
4,gpt2,gpt2_seed_stability,500,42,20,a0.h3,attn,0,3.0,patient_token,clean,0.428718,0.304104,500,0.428718,0.428718,0.000000,animate_pushing


,model_slug,run_name,sample_size,seed,component,component_type,layer,head,position_label,status,example_count,pca_explained_variance_1,pca_explained_variance_2,clean_corrupt_centroid_distance,nearest_centroid_accuracy,paired_delta_count,delta_vector_norm_mean,delta_vector_norm_std
0,gpt2,gpt2_seed_stability,500,42,a0.h3,attn,0,3.0,by_token,ok,1000,0.394050,0.121167,2.239354e+00,0.789,500,3.658083,1.392972
1,gpt2,gpt2_seed_stability,500,42,a0.h3,attn,0,3.0,final_token,ok,1000,0.224392,0.112652,2.646129e-01,0.825,500,0.621401,0.179245
2,gpt2,gpt2_seed_stability,500,42,a0.h3,attn,0,3.0,patient_token,ok,1000,0.092663,0.077687,1.705984e-08,0.500,500,0.000000,0.000000
3,gpt2,gpt2_seed_stability,500,42,a0.h3,attn,0,3.0,verb_token,ok,1000,0.156390,0.102279,2.456409e+00,0.845,500,5.889781,0.648147
4,gpt2,gpt2_seed_stability,500,42,a0.h3,attn,0,3.0,was_token,ok,1000,0.149031,0.103829,1.507887e-09,0.500,500,0.000000,0.000000


## Activation Result Plots

These plots are rendered from the precomputed GPU artifact bundle. They do not run model forward passes.


In [27]:
from __future__ import annotations

def slot_label_from_cols(frame: pd.DataFrame) -> pd.Series:
    return (
        frame["model_slug"].astype(str)
        + " / " + frame["run_name"].astype(str)
        + " / s" + frame["sample_size"].astype(str)
        + " seed " + frame["seed"].astype(str)
    )


def first_available(values: list[str], columns: pd.Index) -> list[str]:
    return [value for value in values if value in columns]


if loaded_activation_results_dir is not None:
    print(f"Plotting activation results from {loaded_activation_results_dir}")

if not attention_semantics.empty:
    attention_plot = attention_semantics.copy()
    attention_plot["key_position_label"] = attention_plot["key_position_label"].replace({"final": "the"})
    attention_order = ["BOS", "The", "patient", "was", "verb", "by", "the"]
    attention_plot = (
        attention_plot.groupby(["model_slug", "condition", "key_position_label"], dropna=False)
        .agg(
            attention_mass_mean=("attention_mass_mean", "mean"),
            attention_mass_std=("attention_mass_std", "mean"),
            example_count=("example_count", "mean"),
        )
        .reset_index()
    )
    attention_index = pd.MultiIndex.from_product(
        [
            sorted(attention_plot["model_slug"].unique()),
            sorted(attention_plot["condition"].unique()),
            attention_order,
        ],
        names=["model_slug", "condition", "key_position_label"],
    )
    attention_plot = (
        attention_plot.set_index(["model_slug", "condition", "key_position_label"])
        .reindex(attention_index)
        .reset_index()
    )
    attention_plot["observed"] = attention_plot["attention_mass_mean"].notna()
    attention_fig = px.bar(
        attention_plot,
        x="key_position_label",
        y="attention_mass_mean",
        color="condition",
        barmode="group",
        facet_col="model_slug",
        facet_col_wrap=2,
        hover_data=first_available(["attention_mass_std", "example_count", "observed"], attention_plot.columns),
        category_orders={"key_position_label": attention_order},
        title="Necessary attention heads: mean attention from final 'the' to sentence positions",
    )
    attention_fig.update_yaxes(matches=None)
    attention_fig.update_layout(xaxis_title="Key position", yaxis_title="Mean attention mass")
    show_fig(attention_fig, height=760)
else:
    print("No attention_semantics rows available to plot.")

if not readout_semantics.empty:
    readout_plot = readout_semantics.copy()
    readout_plot["slot"] = slot_label_from_cols(readout_plot)
    readout_plot["component_label"] = readout_plot["component"].astype(str) + " @ " + readout_plot["position_label"].astype(str)
    readout_top = (
        readout_plot.drop_duplicates(["model_slug", "run_name", "sample_size", "seed", "component", "position_label"])
        .assign(abs_delta=lambda rows: rows["clean_minus_corrupt_readout"].abs())
        .sort_values(["model_slug", "abs_delta"], ascending=[True, False], kind="stable")
        .groupby("model_slug", group_keys=False)
        .head(18)
    )
    readout_fig = px.bar(
        readout_top,
        x="component_label",
        y="clean_minus_corrupt_readout",
        color="component_type",
        facet_col="model_slug",
        facet_col_wrap=2,
        hover_data=first_available(["readout_direction", "clean", "corrupt", "layer", "head", "example_count"], readout_top.columns),
        title="Largest clean-minus-corrupt animate readout deltas by component and position",
    )
    readout_fig.add_hline(y=0, line_width=1, line_color="black")
    readout_fig.update_layout(xaxis_tickangle=-45, xaxis_title="Component @ position", yaxis_title="Clean minus corrupt readout")
    readout_fig.update_yaxes(matches=None)
    show_fig(readout_fig, height=860)
else:
    print("No readout_semantics rows available to plot.")

if not activation_geometry_summary.empty:
    geometry_ok = activation_geometry_summary[activation_geometry_summary["status"] == "ok"].copy()
    if not geometry_ok.empty:
        geometry_top = (
            geometry_ok.sort_values(["model_slug", "clean_corrupt_centroid_distance"], ascending=[True, False], kind="stable")
            .groupby("model_slug", group_keys=False)
            .head(30)
        )
        geometry_top["component_label"] = geometry_top["component"].astype(str) + " @ " + geometry_top["position_label"].astype(str)
        geometry_fig = px.scatter(
            geometry_top,
            x="clean_corrupt_centroid_distance",
            y="nearest_centroid_accuracy",
            color="component_type",
            symbol="position_label",
            facet_col="model_slug",
            facet_col_wrap=2,
            hover_name="component_label",
            hover_data=first_available(["pca_explained_variance_1", "pca_explained_variance_2", "delta_vector_norm_mean", "example_count"], geometry_top.columns),
            title="Activation geometry: clean/corrupt separability for strongest component-position pairs",
        )
        geometry_fig.update_xaxes(matches=None)
        geometry_fig.update_yaxes(matches=None, range=[0, 1.02])
        geometry_fig.update_layout(xaxis_title="Centroid distance", yaxis_title="Nearest-centroid accuracy")
        show_fig(geometry_fig, height=760)
    else:
        display(activation_geometry_summary["status"].value_counts().rename_axis("status").reset_index(name="rows"))
else:
    print("No activation_geometry_summary rows available to plot.")

if not activation_pca_projection.empty and not activation_geometry_summary.empty:
    geometry_ok = activation_geometry_summary[activation_geometry_summary["status"] == "ok"].copy()
    top_pairs = (
        geometry_ok.sort_values(["model_slug", "clean_corrupt_centroid_distance"], ascending=[True, False], kind="stable")
        .groupby("model_slug", group_keys=False)
        .head(1)[["model_slug", "run_name", "sample_size", "seed", "component", "position_label"]]
    )
    pca_plot = activation_pca_projection.merge(top_pairs, on=["model_slug", "run_name", "sample_size", "seed", "component", "position_label"], how="inner")
    pca_plot = (
        pca_plot.groupby(["model_slug", "run_name", "sample_size", "seed", "component", "position_label", "condition"], group_keys=False)
        .head(500)
        .copy()
    )
    if not pca_plot.empty:
        pca_plot["panel"] = pca_plot["model_slug"].astype(str) + " | " + pca_plot["component"].astype(str) + " @ " + pca_plot["position_label"].astype(str)
        pca_fig = px.scatter(
            pca_plot,
            x="pc1",
            y="pc2",
            color="condition",
            facet_col="panel",
            facet_col_wrap=2,
            opacity=0.55,
            title="PCA projections for the most separable necessary component-position pair per model",
        )
        pca_fig.update_traces(marker={"size": 5})
        pca_fig.update_xaxes(matches=None)
        pca_fig.update_yaxes(matches=None)
        pca_fig.update_layout(xaxis_title="PC1", yaxis_title="PC2")
        show_fig(pca_fig, height=860)
    else:
        print("No PCA projection rows matched the selected top geometry pairs.")
else:
    print("No activation_pca_projection rows available to plot.")


Plotting activation results from /gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/eap_ig_localization/necessary_semantics_activations_2026-06-21


## Cross-Model Semantic Comparison

This section compares semantic signatures, not exact component identities. Static role/component summaries are always available; activation-derived columns populate after the optional activation run.


In [28]:
from __future__ import annotations
def top_role_text(rows: pd.DataFrame, role: str, limit: int = 4) -> str:
    subset = rows[rows["edge_role"] == role].sort_values("abs_score", ascending=False, kind="stable")
    if subset.empty:
        return "none"
    return ", ".join(subset["collapsed_edge"].astype(str).head(limit))


def top_component_text(rows: pd.DataFrame, component_type: str | None = None, limit: int = 4) -> str:
    subset = rows if component_type is None else rows[rows["component_type"] == component_type]
    subset = subset.sort_values("abs_incident_mass", ascending=False, kind="stable")
    if subset.empty:
        return "none"
    return ", ".join(subset["component"].astype(str).head(limit))


semantic_component_cards = component_cards.copy()
if not semantic_component_cards.empty and not attention_semantics.empty:
    top_attention = (
        attention_semantics.sort_values("attention_mass_mean", ascending=False, kind="stable")
        .groupby(SLOT_COLUMNS + ["component"], dropna=False)
        .first()
        .reset_index()
    )
    top_attention["attention_target_summary"] = top_attention["key_position_label"].astype(str) + " (" + top_attention["condition"].astype(str) + ", mass=" + top_attention["attention_mass_mean"].round(4).astype(str) + ")"
    semantic_component_cards = semantic_component_cards.drop(columns=["attention_target_summary"], errors="ignore").merge(
        top_attention[SLOT_COLUMNS + ["component", "attention_target_summary"]],
        on=SLOT_COLUMNS + ["component"],
        how="left",
    )
if not semantic_component_cards.empty and not readout_semantics.empty and "readout_direction" in readout_semantics.columns:
    final_readout = (
        readout_semantics[readout_semantics["position_label"] == "final_token"]
        .groupby(SLOT_COLUMNS + ["component"], dropna=False)
        .agg(readout_direction=("readout_direction", lambda values: values.dropna().iloc[0] if len(values.dropna()) else "unknown"))
        .reset_index()
    )
    semantic_component_cards = semantic_component_cards.drop(columns=["readout_direction"], errors="ignore").merge(
        final_readout,
        on=SLOT_COLUMNS + ["component"],
        how="left",
    )
if not semantic_component_cards.empty:
    semantic_component_cards["attention_target_summary"] = semantic_component_cards["attention_target_summary"].fillna("activation_not_run")
    semantic_component_cards["readout_direction"] = semantic_component_cards["readout_direction"].fillna("activation_not_run")

interpretation_rows = []
for key, edges in semantic_edges.groupby(SLOT_COLUMNS, dropna=False):
    slot_dict = dict(zip(SLOT_COLUMNS, key))
    comps = semantic_component_cards.merge(pd.DataFrame([slot_dict]), on=SLOT_COLUMNS, how="inner") if not semantic_component_cards.empty else pd.DataFrame()
    selected_budget = int(edges["selected_budget"].iloc[0])
    interpretation_rows.append(
        {
            **slot_dict,
            "selected_budget": selected_budget,
            "pickup": top_role_text(edges, "input_to_component"),
            "transfer": "; ".join(text for text in [top_role_text(edges, "head_to_mlp"), top_role_text(edges, "mlp_to_mlp")] if text != "none") or "none",
            "processing": top_component_text(comps, "mlp"),
            "readout": top_role_text(edges, "component_to_logits"),
            "top_heads": top_component_text(comps, "attn"),
            "top_mlps": top_component_text(comps, "mlp"),
        }
    )

cross_model_semantic_summary = pd.DataFrame(interpretation_rows).sort_values(SLOT_COLUMNS, kind="stable")

def add_activation_rollups(summary_frame: pd.DataFrame) -> pd.DataFrame:
    rows = summary_frame.copy()
    if not readout_semantics.empty:
        readout_rollup = (
            readout_semantics[readout_semantics["position_label"] == "final_token"]
            .groupby(SLOT_COLUMNS + ["component_type"], dropna=False)
            .agg(mean_final_readout=("readout_mean", "mean"))
            .reset_index()
            .pivot_table(index=SLOT_COLUMNS, columns="component_type", values="mean_final_readout", aggfunc="first")
            .reset_index()
            .rename(columns={"attn": "mean_head_final_readout", "mlp": "mean_mlp_final_readout"})
        )
        rows = rows.merge(readout_rollup, on=SLOT_COLUMNS, how="left")
    if not activation_geometry_summary.empty and "clean_corrupt_centroid_distance" in activation_geometry_summary.columns:
        geometry_rollup = (
            activation_geometry_summary[activation_geometry_summary["status"] == "ok"]
            .groupby(SLOT_COLUMNS + ["component_type"], dropna=False)
            .agg(mean_centroid_distance=("clean_corrupt_centroid_distance", "mean"))
            .reset_index()
            .pivot_table(index=SLOT_COLUMNS, columns="component_type", values="mean_centroid_distance", aggfunc="first")
            .reset_index()
            .rename(columns={"attn": "mean_head_pca_separability", "mlp": "mean_mlp_pca_separability"})
        )
        rows = rows.merge(geometry_rollup, on=SLOT_COLUMNS, how="left")
    return rows

cross_model_semantic_summary = add_activation_rollups(cross_model_semantic_summary)

display(cross_model_semantic_summary)

cross_role_distribution = (
    edge_role_summary.groupby(["model_slug", "edge_role"], dropna=False)
    .agg(mean_edge_count=("edge_count", "mean"), mean_abs_score=("total_abs_score", "mean"))
    .reset_index()
)
cross_component_distribution = (
    component_type_summary.groupby(["model_slug", "component_type"], dropna=False)
    .agg(mean_component_count=("component_count", "mean"), mean_abs_incident_mass=("total_abs_incident_mass", "mean"), mean_layer=("mean_layer", "mean"))
    .reset_index()
)

display(cross_role_distribution)
display(cross_component_distribution)


,model_slug,run_name,sample_size,seed,selected_budget,pickup,transfer,processing,readout,top_heads,top_mlps,mean_head_final_readout,mean_mlp_final_readout,mean_head_pca_separability,mean_mlp_pca_separability
0,Qwen_Qwen3-4B,qwen_seed_stability,500,42,20,"input->m0, input->a0.h18, input->a0.h15, input...","a0.h18->m0; m0->m1, m0->m3, m3->m5, m0->m5","m0, m3, m5, m28","m28->logits, m32->logits, m31->logits, m33->lo...","a0.h18, a0.h15, a0.h9, a0.h22","m0, m3, m5, m28",-0.001554,0.160648,0.044161,12.081643
1,google_gemma-3-4b-pt,gemma_seed_stability,500,42,20,"input->m0, input->a3.h0, input->m5, input->m3","a0.h5->m0; m5->m6, m3->m4","m0, m5, m3, m33","m33->logits, m28->logits, m22->logits, m25->lo...","a3.h0, a0.h5","m0, m5, m3, m33",0.676502,30.083516,5.776336,687.179576
2,gpt2,gpt2_seed_stability,500,42,20,"input->m0, input->a0.h4, input->a0.h3","m0->m1, m0->m3, m0->m2, m6->m7","m0, m8, m1, m10","m10->logits, m9->logits, a9.h3->logits, m8->lo...","a9.h3, a0.h4, a8.h7, a0.h3","m0, m8, m1, m10",0.006428,0.358804,1.545566,8.478960
3,meta-llama_Llama-3.2-3B,llama_seed_stability,500,42,20,"input->m0, input->m1, input->m2","m0->m1, m0->m2, m2->m3, m1->m2","m0, m2, m1, m27","m27->logits, a27.h0->logits, m19->logits, m26-...","a27.h0, a1.h3, a27.h13, a3.h18","m0, m2, m1, m27",-0.007434,0.010310,0.576016,2.223607


,model_slug,edge_role,mean_edge_count,mean_abs_score
0,Qwen_Qwen3-4B,component_to_logits,6.0,0.012556
1,Qwen_Qwen3-4B,head_to_mlp,1.0,0.001837
2,Qwen_Qwen3-4B,input_to_component,9.0,0.019961
3,Qwen_Qwen3-4B,mlp_to_mlp,4.0,0.007556
4,google_gemma-3-4b-pt,component_to_logits,11.0,0.014283
5,google_gemma-3-4b-pt,head_to_mlp,1.0,0.000895
6,google_gemma-3-4b-pt,input_to_component,6.0,0.010910
7,google_gemma-3-4b-pt,mlp_to_mlp,2.0,0.001872
8,gpt2,component_to_logits,9.0,0.766579
9,gpt2,input_to_component,3.0,0.863247


,model_slug,component_type,mean_component_count,mean_abs_incident_mass,mean_layer
0,Qwen_Qwen3-4B,attn,8.0,0.017354,0.000000
1,Qwen_Qwen3-4B,mlp,10.0,0.033949,18.500000
2,google_gemma-3-4b-pt,attn,2.0,0.002589,1.500000
3,google_gemma-3-4b-pt,mlp,17.0,0.028137,18.058824
4,gpt2,attn,9.0,0.664406,5.000000
5,gpt2,mlp,10.0,1.916787,5.700000
6,meta-llama_Llama-3.2-3B,attn,5.0,0.073724,16.600000
7,meta-llama_Llama-3.2-3B,mlp,11.0,0.500032,15.090909


## Optional CSV Export


In [29]:
from __future__ import annotations
tables_for_export = {
    "edge_role_summary": edge_role_summary,
    "component_semantic_cards": semantic_component_cards,
    "attention_semantics": attention_semantics,
    "readout_semantics": readout_semantics,
    "activation_geometry_summary": activation_geometry_summary,
    "activation_pca_projection": activation_pca_projection,
    "target_metadata": target_metadata,
    "cross_model_semantic_summary": cross_model_semantic_summary,
}

if EXPORT_CSV:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    for name, frame in tables_for_export.items():
        if isinstance(frame, pd.DataFrame) and not frame.empty:
            frame.to_csv(EXPORT_DIR / f"{name}.csv", index=False)
    print(f"Exported semantic-analysis tables to {EXPORT_DIR}")
else:
    print("CSV export disabled. Set EXPORT_CSV = True in the configuration cell to write semantic-analysis tables.")


CSV export disabled. Set EXPORT_CSV = True in the configuration cell to write semantic-analysis tables.


## Static Assertions


In [30]:
from __future__ import annotations
assert selected_summary["selected_budget"].notna().all(), "Every selected slot must record selected_budget."
assert not semantic_edges.empty, "Edge semantic table is empty."
assert set(semantic_edges["edge_role"]).issubset({"input_to_component", "head_to_mlp", "mlp_to_mlp", "component_to_logits", "other"})
assert component_cards.empty or component_cards["component_type"].isin(["attn", "mlp", "other"]).all()
assert hook_specs.empty or hook_specs["component"].is_unique

print("Static notebook checks passed.")
print(f"Selected slots: {len(selected_summary)}")
print(f"Collapsed necessary edges: {len(semantic_edges)}")
print(f"Necessary non-structural components: {0 if component_cards.empty else component_cards['component'].nunique()}")
print(f"Default activation sample size: {N_EXAMPLES}")


Static notebook checks passed.
Selected slots: 4
Collapsed necessary edges: 80
Necessary non-structural components: 48
Default activation sample size: 500
